# 🍛 CulinaryVLM — Fine-tuned Model Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivam697/CulinaryVLM/blob/main/notebooks/evaluate_vlm.ipynb)

This notebook verifies the fine-tuned **Llama-3.2-11B-Vision** QLoRA adapter hosted at:
[`shivamminde/culinary-vlm-qlora`](https://huggingface.co/shivamminde/culinary-vlm-qlora)

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

---
**Model details:**
- Base: `unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit`  
- Adapter: QLoRA (r=16, alpha=32) on language layers only  
- Vision layers: frozen → **text-in, text-out** at inference  
- Input format: `Category + Context + Question → Answer`

## Step 1 — Install Dependencies

In [ ]:
# Install Unsloth (same library used for training) + HF Hub
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q huggingface_hub
print('✅ Dependencies installed')

## Step 2 — Authenticate with Hugging Face

Get your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login
import os

# Option A: paste token directly (not recommended for sharing)
# login(token="hf_YOUR_TOKEN_HERE")

# Option B: use Colab secrets (recommended)
# Go to the key icon (🔑) in the left panel → add secret 'HF_TOKEN'
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ Logged in via Colab secret')
except Exception:
    # Fallback: interactive login
    login()
    print('✅ Logged in interactively')

## Step 3 — Load the Fine-Tuned Model

Unsloth automatically loads the LoRA adapter on top of the base model.

In [ ]:
from unsloth import FastVisionModel
import torch

ADAPTER_ID = "shivamminde/culinary-vlm-qlora"

print(f"Loading {ADAPTER_ID} ...")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=ADAPTER_ID,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

# Switch to inference mode (disables dropout, enables faster kernels)
FastVisionModel.for_inference(model)
print('\n✅ Model loaded and ready for inference!')

## Step 4 — Single Query Inference

Test the model on a custom question. Edit `category`, `context`, and `question` below.

In [ ]:
def ask_model(category: str, question: str, context: str = "") -> str:
    """Run inference using the training input format."""
    parts = [f"Category: {category}"]
    if context:
        parts.append(f"Context: {context}")
    parts.append(f"Question: {question}")
    user_content = "\n\n".join(parts)

    messages = [{"role": "user", "content": user_content}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return answer.strip()


# ── Example 1: With context (segment-grounded QA) ──────────────────────────
answer = ask_model(
    category="Hyderabadi",
    context="Action: Adding turmeric powder. Description: The person is adding turmeric powder to the marinated chicken in the bowl.",
    question="What ingredients are being used in this step?"
)
print("Q: What ingredients are being used in this step?")
print(f"A: {answer}")
print()

# ── Example 2: General knowledge (no context) ───────────────────────────────
answer2 = ask_model(
    category="Kolkata",
    question="What makes Kolkata biryani different from Hyderabadi biryani?"
)
print("Q: What makes Kolkata biryani different from Hyderabadi biryani?")
print(f"A: {answer2}")

## Step 5 — Batch Evaluation on Test Set

Upload `datasets/qa/test_filtered.json` from the repo, or clone the repo directly.

In [ ]:
import os

if not os.path.exists("test_filtered.json"):
    # Clone just the test file from the GitHub repo
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/shivam697/CulinaryVLM.git
    !cd CulinaryVLM && git sparse-checkout set datasets/qa
    !cp CulinaryVLM/datasets/qa/test_filtered.json test_filtered.json
    print('✅ test_filtered.json downloaded from GitHub')
else:
    print('✅ test_filtered.json already present')

In [ ]:
import json

# Load test set
with open("test_filtered.json") as f:
    raw = json.load(f)

test_data = [
    qa for qa in raw.get("qa_pairs", [])
    if not qa.get("needs_generation")
    and qa.get("question", "").strip()
    and qa.get("answer", "").strip()
]
print(f"Valid test samples: {len(test_data)}")

# Run inference on first 20 samples
N = 20
correct_keywords = 0

for i, qa in enumerate(test_data[:N]):
    category = qa.get("category", "biryani")
    context  = qa.get("context", "")
    question = qa.get("question", "")
    expected = qa.get("answer", "")

    generated = ask_model(category=category, context=context, question=question)

    # Simple keyword overlap check
    expected_words = set(expected.lower().split())
    generated_words = set(generated.lower().split())
    overlap = len(expected_words & generated_words) / max(len(expected_words), 1)
    if overlap >= 0.3:
        correct_keywords += 1

    print(f"\n[{i+1}/{N}] Category: {category}")
    print(f"  Q: {question[:80]}")
    print(f"  Expected : {expected[:100]}")
    print(f"  Generated: {generated[:100]}")
    print(f"  Overlap  : {overlap:.0%}")

print(f"\n{'='*60}")
print(f"Keyword overlap ≥30% on {correct_keywords}/{N} samples ({correct_keywords/N:.0%})")

---

## 🎉 Done!

You've verified the fine-tuned CulinaryVLM adapter.

**Live demo:** [culinary-vlm.vercel.app](https://culinary-vlm.vercel.app)  
**API:** [culinary-vlm-api.onrender.com](https://culinary-vlm-api.onrender.com)  
**Model:** [shivamminde/culinary-vlm-qlora](https://huggingface.co/shivamminde/culinary-vlm-qlora)  
**Repo:** [github.com/shivam697/CulinaryVLM](https://github.com/shivam697/CulinaryVLM)